# Instalando as dependências necessárias                                           

In [ ]:
%pip install astropy matplotlib numpy
%pip install opencv-python

# Importando as bibliotecas usadas

In [1]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
from astropy.table import Table
from astropy.units import UnitsWarning
from astropy.visualization import make_lupton_rgb
import warnings
import matplotlib.patches as patches
import os
from pathlib import Path
import pandas as pd
import cv2

# AUXILIARY FUNCTIONS

In [2]:
def is_in_stamp(x0,x1,y0,y1,x,y):
        """
        Essa função verifica se um objeto com cordenadas (w1,w2,z1,z2) está completamente dentro de um stamp, com cordenadas dos extremos providênciadas
        Args:
            x0,x1,y0,y1: Cordenadas extremas do stamp
            x,y: cordenadas do centro do objeto
        Returns:
            True se estiver, senão False
        """
        if x0<=x<=x1  and y0<=y<=y1:
            return True
        return False
def is_there_galaxies_in_stamp(x0,x1,y0,y1,cat):
    """
    Essa função verifica se alguma galáxia Rex ou Dev do nosso catálogo está presente no stamp
    Args:
        x0,x1,y0,y1: cordenadas do nosso recorte
        cat: Catalogo filtrado
    Returns:
        true, se existir galáxia rex ou dev
        false, se não existir
    """
    thereis = False
    for  i in range(len(cat)):
        x = int(cat['bx'][i]) # cordenada x e y eo shape da galaxia i
        y = int(cat['by'][i])
        if (is_in_stamp(x0,x1,y0,y1,x,y)):
           thereis = True
    return thereis
    
    

def get_size_biggest_galaxy(cat):
    """
        dado o catalogo filtrado das galáxias, retorna o tamanho em pixeis da maior galáxia 
    """

    size_biggest_galaxy = cat["shape_r"][0]/0.262
    for i in range(len(cat)):
        sizeI = cat["shape_r"][i]/0.262 
        if sizeI>size_biggest_galaxy:
            size_biggest_galaxy = sizeI
    return size_biggest_galaxy
    

def define_class(type):
    """
    Essa função define, com base no tipo de objeto, qual classe ele representa
    """
    if type=="REX":
        return 0
    elif type=="DEV":
        return 1
    else:
        raise ValueError("Type inválido")

def save_stamp_tiff(stamp,outdir,field,idObject):
    """
    Salva um stamp no diretório alvo com padrão tiff (aceita mais de 3 canais), no modelo: survey_dataRelease_field_idobjectStamp_type
    Args:
        stamp: imagem da stamp
        outdir: diretório alvo do stamp
        field: campo da imagem
        idObject: id do objeto central do stamp
        type: tipo do objeto
    """
    # Cria o diretório se não existir
    os.makedirs(outdir, exist_ok=True)
    # Monta o caminho completo
    filepath = os.path.join(outdir, f"legacy_dr10_{field}_{idObject}.tiff")
    # Salva o arquivo. O OpenCV lida com os 4 canais automaticamente.
    success = cv2.imwrite(filepath, stamp)
    if not success:
        print(f"Erro ao salvar: {filepath}")



def save_stamp_png(stamp,outdir,field,idObject):
    """
    Salva um stamp no diretório alvo, no modelo: survey_dataRelease_field_idobjectStamp_type
    Args:
        stamp: imagem da stamp
        outdir: diretório alvo do stamp
        field: campo da imagem
        idObject: id do objeto central do stamp
        type: tipo do objeto
    """
    plt.imsave(
        os.path.join(outdir, f"legacy_dr10_{field}_{idObject}.png"),
        stamp, cmap='gray'
    )

def write_file_bounding_boxes(field,outdir,idObject,bounding_boxes,w,h):
    """
    Escreve os dados da bounding box em um arquivo .txt, padronizado para o YOLO
    Args:
        field: nome do campo
        outdir: diretorio de destino
        idObject: id do objeto
        bounding_boxes: vetor das boundig boxes do campo
        w,h: lagruta e altura do recorte, respectivamente
    
    """
    with open(f"{outdir}/legacy_dr10_{field}_{idObject}.txt","w") as f:
        for box in bounding_boxes:
            class_id,height,width,x_center,y_center = box
            x_center /= w
            y_center /= h
            width /= w
            height /=h
            f.write(f"{class_id} {x_center} {y_center} {width} {height}\n")

def balance_classes_cat(cat_original, prop=3):
    # 1. FILTRAGEM DE COLUNAS: Identifica apenas as colunas simples (1D)
    # Isso remove as colunas problemáticas como apflux, lc_flux, etc.
    colunas_simples = [name for name in cat_original.colnames if len(cat_original[name].shape) <= 1]
    # 2. CONVERSÃO LEVE: Converte para o Pandas apenas com as colunas permitidas
    # Criamos um índice temporário '_id_original' para rastrear as linhas
    df_pandas = cat_original[colunas_simples].to_pandas()
    
    df_pandas['_id_original'] = range(len(cat_original))
    
    # 3. Separar o catálogo por tipo usando o Pandas
    df_dev = df_pandas[df_pandas['type'] == b'DEV']
    df_rex = df_pandas[df_pandas['type'] == b'REX']
    # 4. Descobrir quantas REX podemos ter
    qtd_dev = len(df_dev)
    qtd_rex_maxima = int(qtd_dev * prop)
    # 5. Se houver mais REX do que o limite, fazemos o undersampling aleatório
    if len(df_rex) > qtd_rex_maxima:
        df_rex_filtrado = df_rex.sample(n=qtd_rex_maxima, random_state=42)
    else:
        df_rex_filtrado = df_rex
        
    # 6. Juntar as duas partes de volta no Pandas
    cat_balanceado_df = pd.concat([df_dev, df_rex_filtrado])
    # 7. Misturar o catálogo para os tipos ficarem espalhados
    cat_balanceado_df = cat_balanceado_df.sample(frac=1, random_state=42)
    
    # 8. O TRUQUE: Pegamos a lista de índices das linhas que sobreviveram ao corte
    indices_sobreviventes = cat_balanceado_df['_id_original'].tolist()
    
    # 9. RETORNO: Filtramos o catálogo ORIGINAL do Astropy (que mantém todas as colunas intactas)
    # usando apenas os índices das linhas selecionadas.
    return cat_original[indices_sobreviventes]

# IMAGE PROCESSING FUNCTIONS

In [ ]:
def standardization(imgM):
    """
    Padroniza os pixeis de uma imagem, de modo que, media=0 e dp=1

    Args:
        imgM: matriz da imagem

    Retruns:
        Imagem padronizada
    """
    mean = imgM.mean()
    std = imgM.std()
    return (imgM-mean)/std

def asinh_transformation(flux_array, b_param):
    a = 1.08574
    asinh_mag = a*(np.arcsinh(flux_array / (2 *b_param))+np.exp(b_param))
    return asinh_mag


def img_to_8bits(img):
    """
    Transforma a imagem para 8 bits usando percentis fixos (1% e 99.9%). Isso impede que o ruído de fundo 
    seja artificialmente amplificado em recortes vazios.
    """
    # Substitui NaNs por 0 para não quebrar o cálculo dos percentis
    img_clean = np.nan_to_num(img_clean, nan=0.0)
    vmin = np.percentile(img_clean, 1.0)
    vmax = np.percentile(img_clean, 99.9)    
    if vmax == vmin:  # Evita divisão por zero caso a imagem seja perfeitamente plana
        return np.zeros_like(img_clean, dtype=np.uint8)
        
    img_norm = (img_clean - vmin) / (vmax - vmin)
    # Garante que os valores fiquem estritamente entre 0 e 1 antes de converter para 255
    img_norm = np.clip(img_norm, 0, 1)
    img_255 = img_norm * 255
    return img_255.astype(np.uint8)


def field_to_mult_spectral(brickname,prefix):
    """
     Transforma a imagem do campo em uma img multiespectral usando os filtros R,Z,G e I
     Args:
         brickmame: nome do field
         prefix: prefixo do field
    returns:
        img com 4 canais
    """
    imgR = fits.getdata(f"./data/raw/images/dr10/south/{prefix}/{brickname}/legacysurvey-{brickname}-image-r.fits.fz")
    imgI =  fits.getdata(f"./data/raw/images/dr10/south/{prefix}/{brickname}/legacysurvey-{brickname}-image-i.fits.fz")
    imgG  =  fits.getdata(f"./data/raw/images/dr10/south/{prefix}/{brickname}/legacysurvey-{brickname}-image-g.fits.fz")
    imgZ  =  fits.getdata(f"./data/raw/images/dr10/south/{prefix}/{brickname}/legacysurvey-{brickname}-image-z.fits.fz")
    R = asinh_transformation(imgR,0.012)
    I = asinh_transformation(imgI,0.012) # alica a asihn transformation em todos campos
    G = asinh_transformation(imgG,0.012)
    Z = asinh_transformation(imgZ,0.012)

    # Rs = standardization(R)
    # Is = standardization(I) # padronização dos valores
    # Gs = standardization(G)
    # Zs = standardization(Z)

    mult_spectral = np.zeros((I.shape[0], I.shape[1], 4),dtype=np.uint8) # matriz dos multiespecrtal
    mult_spectral[:,:,0] = img_to_8bits(R) # Vermelho ( trasnforma os valores para 8 bits)
    mult_spectral[:,:,1] = img_to_8bits(I) # Verde
    mult_spectral[:,:,2] = img_to_8bits(G) # Azul
    mult_spectral[:,:,3] = img_to_8bits(Z) # n sei

    return mult_spectral

def field_to_rgb(brickname,prefix):
    """
     Transforma a imagem do campo em um RGB usando os filtros R,I,G
     Args:
         brickmame: nome do field
         prefix: prefixo do field
    returns:
        img rgb
    """
    imgR = fits.getdata(f"./data/raw/images/dr10/south/{prefix}/{brickname}/legacysurvey-{brickname}-image-r.fits.fz")
    imgI =  fits.getdata(f"./data/raw/images/dr10/south/{prefix}/{brickname}/legacysurvey-{brickname}-image-i.fits.fz")
    imgG  =  fits.getdata(f"./data/raw/images/dr10/south/{prefix}/{brickname}/legacysurvey-{brickname}-image-g.fits.fz")
    
    R = asinh_transformation(imgR,0.012)
    I = asinh_transformation(imgI,0.012) # alica a asihn transformation em todos campos
    G = asinh_transformation(imgG,0.012)
    # max_brilho = max(np.max(imgI), np.max(imgR), np.max(imgG))
    # min_brilho = min(np.min(imgI), np.min(imgR), np.min(imgG))

    # factor = max_brilho * 0.5 
    # rgb = make_lupton_rgb(
    # image_r=imgI,  # Atribui o Infravermelho ao Vermelho (mostra poeira/velhas estrelas)
    # image_g=imgR,  # Atribui o Vermelho ao Verde
    # image_b=imgG,  # Atribui o Verde ao Azul
    # stretch=10,      # Controla o contraste geral
    # Q=0.012              # Controla a saturação das cores das galáxias
    # )
    rgb = np.zeros((I.shape[0], I.shape[1], 3),dtype=np.uint8) # matriz dos rgbs
    rgb[:,:,0] = img_to_8bits(I) # Vermelho
    rgb[:,:,1] = img_to_8bits(R) # Verde
    rgb[:,:,2] = img_to_8bits(G) # Azul

    return rgb

# gera_stamps
## Essa função gera os recortes do campo, de modo que um overlap, em pixeis, seja definido para que cada recorte inclua esse overlap como parte da imagem, assim, evitando o caso em que algum objeto é cortado ao meio por um recorte e não seja incluido no próximo.
## Para nosso experimento, foi utilizado um overlap de 72, pois, foi observado que nenhum objeto de interesse tem tamanho maior que 72 pixeis.

In [8]:

def gera_stamps(img, size_cut, overlap, cat, field, goal, mode):
    y0 = 0
    i = 0
    y1 = size_cut
    height_img, width_img = img.shape[:2]
    channels = 3 # quantidade de canais
    # O tamanho final que o YOLO espera
    YOLO_SIZE = 640 
    # Calcula o deslocamento necessário para centralizar o recorte menor dentro do bloco de 640
    pad_offset = (YOLO_SIZE - size_cut) // 2 

    while(y1 <= width_img):
        x0 = 0
        x1 = size_cut        
        while(x1 <= height_img):
            if (x1 + (size_cut - overlap) > width_img) and (x0 < width_img - size_cut):
                x0 = width_img - size_cut 
                x1 = width_img            
                stamp = img[y0:y1, x0:x1] 
                
                # --- APLICANDO O PADDING NO STAMP DA BORDA ---
                stamp_padded = np.zeros((YOLO_SIZE, YOLO_SIZE,channels), dtype=img.dtype)
                stamp_padded[pad_offset:pad_offset+size_cut, pad_offset:pad_offset+size_cut] = stamp
                
                if (mode == "png"):
                    save_stamp_png(stamp_padded, f"data/processed/stamps/images/{goal}/", field, i) 
                else: 
                    save_stamp_tiff(stamp_padded, f"data/processed/stamps/images/{goal}/", field, i) 
                
                # Buscamos as coordenadas locais baseadas no corte menor (size_cut)
                bounding_boxes = build_bounding_boxes_stamp(cat, x0, x1, y0, y1) 
                
                # RECALCULANDO PARA O PADDING: adicionamos o deslocamento nas caixas
                # para que o YOLO saiba que o objeto foi empurrado para o centro do bloco 640
                bounding_boxes_padded = []
                for bbox in bounding_boxes:
                    class_id,h,w, cx, cy = bbox
                    cx_new = (cx + pad_offset) 
                    cy_new = (cy + pad_offset) 
      
                    bounding_boxes_padded.append([class_id,h,w, cx_new, cy_new])

                write_file_bounding_boxes(field, f"data/processed/stamps/labels/{goal}", i, bounding_boxes_padded, YOLO_SIZE, YOLO_SIZE)
                i += 1
                break 

            stamp = img[y0:y1, x0:x1]
            
            # --- APLICANDO O PADDING NO STAMP NORMAL ---
            stamp_padded = np.zeros((YOLO_SIZE, YOLO_SIZE,channels), dtype=img.dtype)
            stamp_padded[pad_offset:pad_offset+size_cut, pad_offset:pad_offset+size_cut] = stamp
            
            if (mode == "png"):
                save_stamp_png(stamp_padded, f"data/processed/stamps/images/{goal}/", field, i) 
            else: 
                save_stamp_tiff(stamp_padded, f"data/processed/stamps/images/{goal}/", field, i) 
            
            bounding_boxes = build_bounding_boxes_stamp(cat, x0, x1, y0, y1) 
            
            # RECALCULANDO PARA O PADDING
            bounding_boxes_padded = []
            for bbox in bounding_boxes:
                class_id,h,w, cx, cy = bbox
                cx_new = (cx + pad_offset) 
                cy_new = (cy + pad_offset) 
  
                bounding_boxes_padded.append([class_id,h,w, cx_new, cy_new])

            write_file_bounding_boxes(field, f"data/processed/stamps/labels/{goal}", i, bounding_boxes_padded, YOLO_SIZE, YOLO_SIZE)
            
            x0 = x1 - overlap
            x1 = x0 + size_cut
            i += 1
            
        if (y1 + (size_cut - overlap) > height_img) and (y0 < height_img - size_cut): 
            y0 = height_img - size_cut
            y1 = height_img
            continue
        y0 = y1 - overlap
        y1 = y0 + size_cut
    return i    

# BOUNDING BOX PROCESSING
## Para gerar as bounding boxes de cada recorte, utilizamos duas funções: gera_bounding_box e build_bounding_boxes_stamp.
## gera_bounding_box é uma função que dado o stamp e mais alguns dados do objeto detalhados no código, constrói a elipse que representa este objeto e, por fim, retorna o menor quadrado ou retângulo que possui a elipse inscrita dentro. Definimos o quadrado caso a magnitude do objeto passe de um limiar (galáxias menos brilhantes), assim, concertando alguns erros de angulação das elipses.
## build_bounding_boxes_stamp é uma função que gera todas bounding boxes das galáxias REX e DEV presentes no recorte, assim, armezenando e retornando os seguintes dados, visando o treinamento pelo YOLO: class_Id, altura, largura, cordenada do centro do objeto em relação a x e y.

In [5]:
def gera_bounding_box(bx,by,shape_r,shape_e1,shape_e2,x0,y0,mag,pixscale=0.262):
    """
    Gera, no stamp, as cordenadas de uma bounding box para o objeto de interesse
    Args:
        bx,by: cordenadas do centro do objeto, na imagem do céu
        shape_r: shape_r do objeto
        shape_e1: shape_e1 do objeto
        shpae_e2: shape_e2 do objeto
        x0: cordenada horizontal de inicio do stamp
        y0: cordenada vertical de inicio do stamp
        snr: sinal ruido do objeto de interesse
    Returns:
        cordenas da caixa bounding box dentro do stamp
    """
    # Se o objeto for muito brilhante, a caixa cresce para pegar o halo
    k = 3 if mag < 8 else 6
    e = np.sqrt(shape_e1**2 + shape_e2**2) # modulo do coeficiente de elipticidade( determina o grau de alongamento da elipse)
    q = (1 - e) / (1 + e) # b/a  onde (b é o semieixo menor da elipse e a é o maior)
    theta = 0.5 * np.arctan2(shape_e2, shape_e1) # definição do ângulo da galáxia do eixo maior em relação ao eixo x do cartesiano
    a = (1-e)*10  #semieixo maior, multiplicado por um fator de escala
    b = (1+e)*10 # definição de b com base em q, multiĺicado por um fator de escala
    t = np.linspace(0, 2*np.pi, 200) #gerando os pontos da elipse
    x = a * np.cos(t)
    y = b * np.sin(t) ## x e y, equação paramétrica da elipse

    cos_t, sin_t = np.cos(theta), np.sin(theta)
    x_rot = x * cos_t - y * sin_t
    y_rot = x * sin_t + y * cos_t # rotaciona theta graus a elipse 
    x_ell = x_rot + bx 
    y_ell = y_rot + by #translada a elipse para o centro da galaxia, já que ela estáva centrada na origem do plano cartesioano
    # recorta a  boundinbox (retangulo menor retangulo que possui a elipse dentro) no contexto do stamp
    xmin_s = x_ell.min() - x0
    xmax_s = x_ell.max() - x0
    ymin_s = y_ell.min() - y0
    ymax_s = y_ell.max() - y0
    if (mag<19):
        return xmin_s, ymin_s, xmax_s, ymax_s ## galáxias mais brilhantes, retorna o menor retangulo
    
    # galáxias menos brilhantes, pegamos o menor quadradado que tem a elipse dentro
    diff_x = xmax_s - xmin_s
    diff_y = ymax_s - ymin_s
    tamanho = max(diff_x, diff_y)
    centro_x, centro_y = (xmin_s + xmax_s)/2, (ymin_s + ymax_s)/2
    xmin_final = centro_x - tamanho/2
    xmax_final = centro_x + tamanho/2
    ymin_final = centro_y - tamanho/2
    ymax_final = centro_y + tamanho/2

    return xmin_final, ymin_final, xmax_final, ymax_final
    
def build_bounding_boxes_stamp(cat,x0,x1,y0,y1):
    """
    Essa função gera todas bounding box de galaxias Rex, DEV presentes no stamp com cordenadas x0, x1, y0, y1.
    Args:
        cat: catalago referente ao campo
        x0, x1, y0, y1: cordenadas extremas referentes ao stamp
        bx, by: cordenadas centrais do objeto principal do stamp
    Returns:
        bounging_boxes: array de arrays, onde cada elemento representa uma bounging box em função do yolo( class_id, height, width, x_center, y_center)
    """
    bounding_boxes = []
    for j in range(len(cat)):
            
            x = int(cat['bx'][j]) # centro do objeto
            y = int(cat['by'][j])
            shape_r = cat['shape_r'][j]
            isObjetctInStamp = is_in_stamp(x0,x1,y0,y1,x,y) # verifica se o objeto está dentro do stamp  
            if  isObjetctInStamp:
                 
                shape_e1 = cat['shape_e1'][j]
                shape_e2 = cat['shape_e2'][j]
                flux_r =  cat['flux_r'][j]
                flux_ivar_r = cat['flux_ivar_r'][j]
                snr = flux_r*np.sqrt(flux_ivar_r)
                type = cat['type'][j]
                xmin,ymin, xmax,ymax = gera_bounding_box(x,y,shape_r,shape_e1,shape_e2,x0,y0,snr) # cordenadas da bounding box


                
                if xmin+x0>= x0 and xmax+x0<=x1 and ymin+y0>= y0 and ymax+y0<=y1: #verifico se a galaxia está INTEIRA dentro do stamp
                    class_id = define_class(type)
                    y_center = (ymax+ymin)/2
                    x_center = (xmax+xmin)/2
                    width = xmax-xmin
                    height = ymax-ymin
                    
                    if width < 10 or height < 10: # não aceita bounding boxes com menos de 8 pixeis( evitando objetos muitop pequenos, pois podem se confundir com background)
                        print("Não aceito essa bounding box")
                        continue
                    #bounding_boxes.append([xmin,ymin,xmax,ymax,type]) #temporario para testes
                    bounding_boxes.append([class_id,height,width,x_center,y_center])
    if len(bounding_boxes)==0:
        print("Esse recorte não possui nenhum objeto presente")
    return bounding_boxes


## Gerando os recortes de treino e validação a partir de todas imagens relacionadas às imagens dos campos presentes no diretório /data/raw/images/dr10/south

In [ ]:

warnings.simplefilter('ignore', UnitsWarning)
qtd_recortes = 0 ## vai contar a quantidade de imagens produzidas
qtd_rex = 0
qtd_dev = 0
base_path = Path('./data/raw/images/dr10/south/')
validation = ["0042m432","0042m472",
         "0042m605","0043m042","0043m050","0043m067","0043m342","0043m387","0043m392","0043m445","0043m487","0043m490","0043m567","0043m570","0043m575",
             "0050m140","0050m142"]
# Iterando pela estrutura: prefixo / brick / arquivos
for prefix_dir in base_path.iterdir():
    if prefix_dir.is_dir():
        prefix = prefix_dir.name
        # Lista as pastas de bricks dentro do prefixo
        for brick_dir in prefix_dir.iterdir():
            if brick_dir.is_dir():
                brick_name = brick_dir.name
                print("Gerando recortes para o campo: "+str(brick_name))
                img = field_to_rgb(brick_name,prefix) # recorte multispectral
                cat = None
                for file in brick_dir.glob('*'): # vai pegar o catlogo agr
                    if 'tractor' in file.name:
                        cat = Table.read(f"./data/raw/images/dr10/south/{prefix}/{brick_name}/{file.name}") # catalogo
                        mask = np.isin(cat['type'], [b'REX', b'DEV']) #transforma o catologo, em um catalogo contendo apenas REX E DEV
                        cat = cat[mask]
                        mask2 = (cat['flux_r'] * np.sqrt(cat['flux_ivar_r'])) > 10 # aceita objetos apenas com snr>10
                        cat = cat[mask2]
                        bit_sub_blob = 2**16
                        mask3 = (cat['maskbits'] & bit_sub_blob) == 0 ## tira os objetos qeu são sobrepostos
                        cat = cat[mask3]
                        balanced_cat = balance_classes_cat(cat)
                if(brick_name not in validation): ## se n for essse campo não é validation
                    i = gera_stamps(img,256,64,balanced_cat,brick_name,"train","png") ## gera os stamps em modo tiff para treina
                    qtd_rex += np.sum(balanced_cat['type'] == b'REX')
                    qtd_dev += np.sum(balanced_cat['type'] == b'DEV')
                    qtd_recortes+=i
                else:
                    i = gera_stamps(img,256,64,balanced_cat,brick_name,"validation","png") ## gera os stamps em modo tiff para validação
                print("Recortes de "+str(brick_name)+" gerados.\n")
print("Quantidade de imagens geradas "+str(qtd_recortes))
print("quantidade total de rex para treinamento "+str(qtd_rex))
print("quantidade total de dev para treinamento"+str(qtd_dev))
